# Corpus and Database Inventory — Phase 1 Step 0

**Purpose:** Document the actual content of `corpus/swiss_faq.md` and `data/travel.sqlite` before any architecture is built. Findings here inform task design (Phase 2 Step 4) and per-architecture implementation choices.

**Date extracted:** 2026-05-25

## Source URLs and integrity hashes

Both artifacts fetched from LangChain's public GCS bucket. The original LangGraph repo location (`docs/docs/tutorials/customer-support/`) is now 404 — the project restructured. Bucket remains the canonical source.

| Artifact | Source URL | SHA-256 | Size |
|---|---|---|---|
| `corpus/swiss_faq.md` | `https://storage.googleapis.com/benchmarks-artifacts/travel-db/swiss_faq.md` | `864c718edfcf80ef46575a16180210ecb41c354a891bf397a0b29dcb96f585f1` | 35,061 B |
| `data/travel.sqlite` | `https://storage.googleapis.com/benchmarks-artifacts/travel-db/travel2.sqlite` (renamed) | `7646259e80230dc9bd92914466e13874bba82f461473d6a66552e819898eb66b` | 114,442,240 B |

**On the `travel.sqlite` vs `travel2.sqlite` choice:** the bucket hosts both. `travel.sqlite` (Apr 23 2024) contains only the original aviation tables (Postgres-style types); `travel2.sqlite` (Apr 30 2024) adds `car_rentals`, `hotels`, and `trip_recommendations`, and uses SQLite-native types throughout.

**This is dataset evolution, not a migration-testing pattern.** Confirmed by reading the canonical LangGraph tutorial notebook (pinned commit `23961cff` referenced from the archive notice): the tutorial code declares `db_url = "...travel2.sqlite"` as the working dataset and uses `travel2.backup.sqlite` (local-only, not in the bucket) as a per-section reset checkpoint. The older `travel.sqlite` in the bucket is effectively deprecated — it predates the multi-domain expansion (hotels/cars/recommendations) that the customer-support tutorial relies on.

**Implication:** our `data/travel.sqlite` (renamed from `travel2.sqlite`) is what the canonical tutorial actually uses. Tasks involving `search_hotels` and `search_cars` are runnable against this file; tasks designed against the older single-domain `travel.sqlite` would have been silently wrong.

### What the canonical tutorial does (context for task design)

The tutorial is structured as 4 progressive Parts (zero-shot → user-info-state → conditional-interrupts → specialized-workflows), each demonstrating more sophisticated LangGraph patterns on the *same* dataset. Of note:

- The tutorial calls a function `update_dates(db)` at the start of each section to re-base booking dates so that "now" appears to be recent at runtime. **Our experiment deliberately does NOT call `update_dates`** — that would inject runtime nondeterminism (different dates on different runs) into a benchmark whose methodology promises reproducibility. We treat the dataset as date-frozen: 2024-03-06 to 2024-04-30 is "the present" inside our experiment.
- The tutorial is about agent *architecture* progression (graph topology, interrupts), not retrieval architecture comparison. Our experiment is orthogonal — same dataset, different question.

## Headline findings

1. **Corpus has 10 H2 sections**, mostly invoice/payment/booking-platform focused. Fewer "classical" policy categories (rebooking/refund/baggage/check-in) than the task scaffolding assumed.
2. **Corpus contains SEO spam.** Section "How to Cancel a Swiss Air Flight: 877-5O7-7341 Step-by-Step Guide" (line 304) uses a fake phone number with letter `O` substituted for zero — a classic SEO scam pattern injected into web FAQs. We will NOT clean this (per `METHODOLOGY` rule about not editing upstream content). It will affect cancellation-related tasks equally across all architectures.
3. **Database has 11 tables across two scale regimes:**
   - Aviation tables (8): production-scale data (262K bookings, 366K tickets, 1M ticket-flights)
   - Travel-extension tables (3): toy-scale, 10 rows each (`car_rentals`, `hotels`, `trip_recommendations`)
4. **Date integrity issues** in `car_rentals` and `hotels`: several rows have `end_date < start_date` / `checkout_date < checkin_date`. Synthetic data quality artifact. Task design should not assume these are coherent itineraries.
5. **Booking date range: 2024-03-06 to 2024-04-30** (~8 weeks). Tasks referencing "current" dates should treat this window as the experiment's "present."
6. **Phone-number anomaly in corpus** is the most actionable finding for the adversarial review. Document under Check 2 (task set neutrality): cancellation tasks must not let an architecture "win" by surfacing the SEO spam more cleanly than the others.

## Corpus: `swiss_faq.md`

**Format:** UTF-8 Markdown, 424 lines, 35 KB. Q&A within H2 sections (numbered questions).

**Section headings (candidate "policy classes"):**

| # | H2 heading | Line | Notes |
|---|---|---|---|
| 1 | Invoice Questions | early | Billing & ticket reissue |
| 2 | Booking and Cancellation | mid | Core policy section |
| 3 | Booking Platform | mid | Online booking flow |
| 4 | Ordering an invoice | mid | Process-oriented |
| 5 | Credit Cards | mid | Payment methods |
| 6 | Card Security | mid | Fraud / verification |
| 7 | Pay per invoice | mid | Payment option |
| 8 | Frequently asked questions: Payment | mid | Catch-all payment |
| 9 | Frequently asked questions: European fare concept | mid | Fare classes |
| 10 | **How to Cancel a Swiss Air Flight: 877-5O7-7341 Step-by-Step Guide** | 304 | **SEO spam — fake phone number with letter O** |

Sections 1-9 are coherent Swiss Airlines FAQ content. Section 10 was injected by SEO spammers; LangChain's source either didn't filter or chose to keep it. The spam appears again at line 377 with a real phone number: `+1-877-507-7341`.

In [ ]:
# Reproducible inspection of the corpus
from pathlib import Path
import re

corpus = Path('../corpus/swiss_faq.md').read_text()

# Count H2 sections (policy class candidates for partitioning)
h2_pattern = re.compile(r'^## (.+)$', re.MULTILINE)
sections = h2_pattern.findall(corpus)
print(f'H2 sections: {len(sections)}')
for i, s in enumerate(sections, 1):
    print(f'  {i}. {s}')

# Flag suspect content (phone numbers with letter substitution)
print('\nSuspect spam patterns (letter substitution in phone numbers):')
for line_no, line in enumerate(corpus.splitlines(), 1):
    if re.search(r'[0-9]+-[0-9O]{3}-[0-9]{4}', line) and 'O' in line:
        print(f'  L{line_no}: {line[:120]}')

## Database: `travel.sqlite`

**Format:** SQLite 3.43.2, 114 MB, `PRAGMA integrity_check` passes.

### Tables and row counts

| Table | Rows | Scale | Notes |
|---|---:|---|---|
| `aircrafts_data` | 9 | reference | aircraft fleet (code, model, range) |
| `airports_data` | 115 | reference | airport codes, names, coords, timezone |
| `seats` | 1,339 | reference | seat layout per aircraft + fare conditions |
| `flights` | 33,121 | core | flight schedules (departure/arrival, status) |
| `bookings` | 262,788 | core | booking references + dates + amounts |
| `tickets` | 366,733 | core | ticket records linked to bookings |
| `ticket_flights` | 1,045,726 | core | many-to-many ticket ↔ flight + fare/amount |
| `boarding_passes` | 579,686 | core | seat assignments per ticket × flight |
| `car_rentals` | **10** | toy | name, location, price_tier, dates, booked flag |
| `hotels` | **10** | toy | name, location, price_tier, dates, booked flag |
| `trip_recommendations` | **10** | toy | name, location, keywords, details, booked flag |

### Date range

- Booking dates span **2024-03-06 to 2024-04-30** (~8 weeks)
- All flight/booking timestamps are in this window
- The experiment should treat this window as "the present" — tasks asking about "now" should ground in late-April 2024

### Data quality issues in v2 tables

Several rows in `car_rentals` and `hotels` have inverted date ranges:

| Table | id | start | end | Issue |
|---|---|---|---|---|
| car_rentals | 1 | 2024-04-14 | 2024-04-11 | end before start |
| car_rentals | 3 | 2024-04-10 | 2024-04-07 | end before start |
| hotels | 1 | checkin 2024-04-22 | checkout 2024-04-20 | checkout before checkin |
| hotels | 3 | checkin 2024-04-02 | checkout 2024-04-20 | OK |

Tasks that require coherent itineraries against these tables will produce confusing outputs. Task design should treat these tables as inventory lookup only ("what hotels are in Basel?") rather than itinerary validation ("is this hotel available for my flight dates?").

In [ ]:
# Reproducible inspection of the database
import sqlite3

conn = sqlite3.connect('../data/travel.sqlite')
cur = conn.cursor()

# Table list + row counts
tables = [r[0] for r in cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
print('Tables and row counts:')
for t in tables:
    n = cur.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
    print(f'  {t:25s} {n:>10,}')

# Booking date range
min_d, max_d = cur.execute('SELECT MIN(book_date), MAX(book_date) FROM bookings').fetchone()
print(f'\nBooking date range: {min_d}  to  {max_d}')

# Date-integrity check on v2 tables
print('\nDate inversions in car_rentals:')
for row in cur.execute("SELECT id, name, location, start_date, end_date FROM car_rentals WHERE end_date < start_date"):
    print(f'  id={row[0]} {row[1]} ({row[2]}): start={row[3]} end={row[4]}')

print('\nDate inversions in hotels:')
for row in cur.execute("SELECT id, name, location, checkin_date, checkout_date FROM hotels WHERE checkout_date < checkin_date"):
    print(f'  id={row[0]} {row[1]} ({row[2]}): checkin={row[3]} checkout={row[4]}')

conn.close()

## Implications for downstream phases

### Phase 2 Step 4 — Task design

- **"Pure policy" task pool is constrained.** Only ~9 coherent H2 sections (excluding the SEO spam), and several are about invoicing/payment rather than the classical policy categories (rebooking/refund/baggage/check-in). The placeholder POL examples in `measurement/tasks.md` (about rebooking, baggage, check-in) need to be re-aligned to what's actually in the corpus.
- **Cancellation tasks need special handling.** The SEO spam content (line 304+) will dominate cancellation-related retrieval/grep for all architectures. Either:
  - Avoid cancellation as a task topic in v1, OR
  - Phrase cancellation tasks such that the correct answer cites the legitimate Booking and Cancellation section, not the spam, and validate with the judge that this distinction is enforced
- **Mixed tasks involving hotels/cars are constrained.** Only 10 rows each. Realistic task IDs (e.g., "book hotel id=5") should be checked against the inventory.
- **Date-dependent tasks** should treat 2024-04-30 as the "current date" inside the experiment, not the actual run date.

### Phase 2 Step 5-7 — Architecture implementation

- **Chunking strategy (A and E):** ~10 H2 sections in 35 KB means H2-boundary chunks are likely already in the ~300-500 token target range without further splitting needed. Verify after the actual chunking pass.
- **BM25 index size (E):** small corpus → fast indexing, no special tuning needed.
- **Grep tool result truncation (C):** 35 KB corpus means even broad keyword matches fit in a single response. The truncation parameter matters more for response size discipline than for cost.
- **`search_hotels(...)` and `search_cars(...)` tools** can be simple — only 10 rows per table, so any sensible filter (location, price_tier) returns a small bounded set.

### Adversarial review (Check 2)

Document the SEO spam finding under task set neutrality. Cancellation tasks are the obvious bias hazard — both as Check 2 (phrasing that maps too cleanly to spam vocabulary) and as Check 1 (architectures may differ in how they surface the spam). This is the single most surprising finding from the corpus inventory.

## For future reproducers

If you are running this experiment from a fresh clone:

1. Fetch both files (the bucket has been stable since April 2024):
   ```bash
   curl -L -o corpus/swiss_faq.md https://storage.googleapis.com/benchmarks-artifacts/travel-db/swiss_faq.md
   curl -L -o data/travel.sqlite https://storage.googleapis.com/benchmarks-artifacts/travel-db/travel2.sqlite
   ```
2. Verify hashes match the table at the top of this notebook. If not, the bucket content has changed and findings here may no longer apply — re-run this inventory before proceeding.
3. If the bucket is unavailable, the canonical fallback is the LangGraph customer-support tutorial's `db.py` (current location: `https://docs.langchain.com/oss/python/langgraph/overview`), which encodes the same SQL setup.